In [47]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from itertools import chain
from collections import Counter

import igraph as ig
import matplotlib.pyplot as plt
import seaborn.objects as so
import seaborn as sns

from utils.pandas_setup import pandas_setup
pandas_setup()

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/econommicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

In [48]:
class SetUp:

    def __init__(self):
        self._setup_db()
        self._setup_cache()
        return
    
    def _setup_db(self):
        self.db = duckdb.connect(MY_DATABASE_FILE)
        self.db.sql("ATTACH IF NOT EXISTS ':memory:'")
        self.db.sql(""" SET memory_limit = '56GB';
                        SET threads = 6;
                        SET preserve_insertion_order = false;
                        SET order_by_non_integer_literal=true;
                        SET enable_progress_bar = true;
                        SET temp_directory = '/home/lc/m/.tmp';
                    """)
        self.db.sql("SHOW ALL TABLES").show()
        return
    
    def _setup_cache(self):
        self.cache = diskcache.Cache(MY_CACHE_FILE, size_limit=16_000_000_000)
        print(f'{self.cache.check() = }')
        print(f'{self.cache.volume() = }')
        return
    
    def name_of_global_obj(self, obj=None):
        for objname, oid in globals().items():
            if oid is obj:
                return objname
            

In [49]:
class CoauthorshipNetwork(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def build_authorship_network(self):
        # get works from a few small, top  journals
        source_list = 'https://openalex.org/S2764736659 https://openalex.org/S7397502'
        sql = f"""
                CREATE OR REPLACE TABLE memory.network AS 
                    SELECT replace(source_id, 'https://openalex.org/', '') AS source_id,
                            -- any_value(source_name),
                            replace(institution_id, 'https://openalex.org/', '') AS institution_id,
                            -- any_value(institution_name)
                            count(work_id) AS weight,
                        FROM works w
                        LEFT JOIN authorships a
                            USING (work_id) 
                            WHERE contains('{source_list}', w.source_id) = true 
                                    AND author_name NOT NULL
                                    AND work_id NOT NULL
                        GROUP BY source_id, institution_id, source_name, institution_name
                        ORDER BY weight DESC, source_name, institution_name
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM memory.network").show()
        self.db.sql("COPY memory.network TO '../DATA/NETWORK_DATA/trial.csv' (HEADER false, DELIMITER '\t')")
        return
    
    def extract_graph(self):
        self.g = ig.Graph.Read_Ncol('../DATA/NETWORK_DATA/trial.csv', names=True)
        print(self.g.get_edgelist())
        print(self.g.es["weight"])
        print(self.g.vs["name"])
        return
    
    def pagerank(self):
        ranks = self.g.pagerank()
        print(ranks)
        return


In [50]:
class CitationNetwork(SetUp):

    def __init__(self, kind=None):
        super().__init__()
        self.kind = kind
        return

    def build_base(self):
        # create a table in memory for each work and the assciated publication and authorship information
        sql = """
                --CREATE OR REPLACE TABLE base AS
                    SELECT DISTINCT 
                            w.work_id,
                            w.publication_year,
                            'source' AS kind,
                            w.source_id AS id,
                            w.source_name AS name,
                            w.host_name AS name_ext,
                            a.author_id,
                            a.author_name,
                            w.institutions_distinct_count
                        FROM works w
                            LEFT JOIN authorships a
                                USiNG (work_id)
                    UNION ALL
                    SELECT DISTINCT 
                                w.work_id,
                            w.publication_year,
                            'institution' AS kind,
                            a.institution_id AS id,
                            a.institution_name AS name,
                            a.country_code AS name_ext,
                            a.author_id,
                            a.author_name,
                            w.institutions_distinct_count
                        FROM works w
                            LEFT JOIN authorships a
                                USiNG (work_id)
                    WHERE contains('article review preprint letter', w.type) = true 
                            AND publication_year >= 2010
                            AND id NOT NULL
                            AND author_id NOT NULL
                    ORDER BY work_id
            """
        self.db.sql(sql).show()
        # self._build_name_dictionaries()
        return
    
    def _build_name_dictionaries(self):
        # build dictionarys to find names associated with ID codes for sources and institutions
        df = self.db.sql("SELECT * FROM memory.base").df()
        self.name = {**dict(zip(df.source_id, df.source_name)), **dict(zip(df.institution_id, df.institution_name))}
        self.name_ext = {**dict(zip(df.source_id, df.host_name)), **dict(zip(df.institution_id, df.country_code))}
        return
    
    def report_base(self):
        df = self.db.sql("SELECT * FROM memory.base").df()
        print(f'{df.shape = }\n{df.head()}')
        print(f'{df.work_id.nunique() = }')
        print(f'{df.source_id.nunique() = }')
        print(f'{df.author_id.nunique() = }')
        print(f'{df.institution_id.nunique() = }')
        print(f'{df.country_code.nunique() = }')                        
        return

    def build_citation_tables(self):
        # build table of citation counts between citer and cited (journals and institutions)
        for citer_kind in ['source', 'institution']:
            print(f'{citer_kind = }')
            sql = f"""
                    CREATE OR REPLACE TABLE citation_network_{citer_kind} AS
                        SELECT c.{citer_kind}_id AS citer,
                                b2.{citer_kind}_id AS cited,
                                count(DISTINCT c.citer_id) AS weights,
                                sum(c.institution_cite_value) AS reweights
                            FROM
                            -- SET UP THE "citer" PART
                            (
                            SELECT c.work_id AS citer_id,
                                    unnest(referenced_works) AS cited_id,
                                    b1.{citer_kind}_id,
                                    b1.{citer_kind}_name,
                                    1.0/b1.institutions_distinct_count AS institution_cite_value
                                FROM cited c
                                LEFT JOIN memory.base b1
                                    USING (work_id)
                            ) c
                            -- SET UP THE "citer" PART
                                LEFT JOIN memory.base b2
                                    ON c.cited_id = b2.work_id
                            WHERE c.{citer_kind}_id NOT NULL 
                                    AND b2.{citer_kind}_id NOT NULL
                        GROUP BY ALL
                """
            self.db.sql(sql)
            self.db.sql(f"COPY citation_network_{citer_kind} TO '../DATA/NETWORK_DATA/citation_network_{citer_kind}.csv' (HEADER true, DELIMITER ',')")
        self._build_both()
        raise SystemExit
        return        

    def _build_both(self):
        sql = """
                SELECT citer, cited, weights FROM citation_network_source
                UNION
                SELECT citer, cited, reweights AS weights FROM citation_network_institution 
                ORDER BY weights DESC
                """
        self.db.sql(sql).show()
        return

    def report_citations(self):
        for citer_kind in ['source', 'institution', 'both']:
            print(f'{citer_kind = }')
            if citer_kind == 'both':
                df1 = pd.read_csv('../DATA/NETWORK_DATA/citation_network_source.csv')
                df2 = pd.read_csv('../DATA/NETWORK_DATA/citation_network_institution.csv')
                df2['weights'] = [int(w) if isinstance(w, float) else 0 for w in df2.reweights]
                df = pd.concat([df1, df2], axis=0)
            else:
                df = pd.read_csv(f'../DATA/NETWORK_DATA/citation_network_{citer_kind}.csv')
                if citer_kind == 'institution':
                    df['weights'] = [int(w) if isinstance(w, float) else 0 for w in df.reweights]
            df = df[df.weights > 0].sort_values('weights', ascending=False)
            print(f'{df.shape = }\n{df.head()}')
            print(f'DISTINCT CITERS {df.citer.nunique() = }')
            print(f'DISTINCT CITEDS {df.cited.nunique() = }')
            cntr = Counter(df.weights)
            print(f'COUNT DISTRIBTION {cntr.most_common(8) = }')
            print(f'COUNT DISTRIBTION {cntr.most_common()[:-8:-1] = }')
            print(f'TOTAL CITATIONS {cntr.total() = }')
        return
    
    def extract_citation_tables(self):
        for citer_kind in ['source', 'institution', 'both']:
                                        # ('both', 'source'), ('both', 'institution'), 
                                        # ('source', 'both'), ('institution', 'both'),
                                        # ('both', 'both)')]:
            print(f'{citer_kind = }')
            # if 'both' not in [citer_kind, cited_kind]?
            self.extract_graph(citer_kind)
            self.run_pagerank(citer_kind)
            self.report_pagerank(citer_kind)
            self.weighted_citation_count(citer_kind)
            # self.plot_model(citer_kind)
        return

    def extract_graph(self, citer_kind):
        if citer_kind == 'both':
            df1 = pd.read_csv('../DATA/NETWORK_DATA/citation_network_source.csv')
            df2 = pd.read_csv('../DATA/NETWORK_DATA/citation_network_institution.csv')
            df2['weights'] = [int(w) if isinstance(w, float) else 0 for w in df2.reweights]
            df_edges = pd.concat([df1, df2], axis=0)
        else:
            df_edges = pd.read_csv(f'../DATA/NETWORK_DATA/citation_network_{citer_kind}.csv')
            if citer_kind == 'institution':
                    df_edges['weights'] = [int(w) if isinstance(w, float) else 0 for w in df_edges.reweights]
        df_edges = df_edges[df_edges.weights >= 50]
        print(f'*** DROP EDGES WITH FEWER THAN 50 CITATIONS {df_edges.shape = }\n{df_edges.head()}')
        self.g = ig.Graph.DataFrame(df_edges, directed=True, use_vids=False)
        print(self.g.get_edgelist())
        summary = ig.summary(self.g, verbosity=0, width=78, edge_list_format='auto', max_rows=99999, print_graph_attributes=False, 
                                          print_vertex_attributes=False, print_edge_attributes=False, full=False)
        print(f'*** SUMMARY OF self.g\n{summary}')
        return
    
    def run_pagerank(self, citer_kind):
        print('pageRank')
        ranks = self.g.pagerank(weights='weights', implementation='power')
        print(f'>> CHECK pageRank  - should sum to unity {sum(ranks) = }')
        rank_max = max(ranks)
        ranks = [10.*r/rank_max for r in ranks]
        self.pagerank = pd.DataFrame(zip(ranks, self.g.vs["name"], self.g.degree()), columns=['pageRank','citer', 'pub_count']).sort_values('pageRank', ascending=False)
        temp = self.pagerank
        self.db.sql(f"CREATE OR REPLACE TABLE econ.pagerank_{citer_kind} AS SELECT * FROM temp")
        return
    
    def report_pagerank(self, citer_kind):        
        self.pagerank['name'] = [self.name.get(id) for id in self.pagerank['citer']]
        self.pagerank['name_ext'] = [self.name_ext.get(id) for id in self.pagerank['citer']]
        self.db.sql(f"SELECT * FROM econ.pagerank_{citer_kind}").show()
        print(f'{self.pagerank.shape = }\n{self.pagerank.head(8)}')
        print(f'\n{self.pagerank.tail(8)}')
        print(f'Sum of publications {self.pagerank['pub_count'].sum() = }')
        print(f'Sum of pageranks {self.pagerank['pageRank'].sum() = }')
        self.db.sql("SELECT count(DISTINCT work_id) AS original_works_count FROM works").show()
        return

    def weighted_citation_count(self, citer_kind):
        self.db.sql("SELECT * FROM memory.base").show()
        sql = """ 
                SELECT work_id AS citer_id,
                        unnest(referenced_works) AS cited_id,
                        author_id, 
                        author_name,
                        source_id,
                        institution_id,
                    FROM memory.base b
                    LEFT JOIN cited c
                        USING (work_id)
                    LIMIT 64
                """
        self.db.sql(sql).show()
        return

        # sql = f"""
        #         CREATE OR REPlACE TABLE econ.citation_counts AS
        #             SELECT count(b2.work_id) AS citation_counts,
        #                     sum(p.pagerank) AS weighted_citation_counts,
        #                     b2.author_id AS cited_id,
        #                     b2.author_name AS cited_name,
        #                 FROM
        #                 (
        #                 SELECT c.work_id AS citer_id, 
        #                         unnest(referenced_works) AS cited_id,
        #                         b1.{citer_kind}_id AS citer,
        #                         b1.source_id
        #                     FROM cited c
        #                     LEFT JOIN memory.base b1
        #                         ON c.work_id = b1.work_id
        #                 ) sub
        #                     LEFT JOIN memory.base b2
        #                         ON sub.cited_id = b2.work_id
        #                         LEFT JOIN pagerank_{citer_kind} p
        #                             ON sub.citer = p.citer
        #                 WHERE p.citer NOT NULL AND b2.author_id NOT NULL
        #             GROUP BY ALL
        #             ORDER BY citation_counts DESC
        #     """ 
        # self.db.sql(sql)
        # self.db.sql("SELECT * FROM econ.citation_counts").show()
        # return
    
    def plot_model(self, citer_kind):
        df = self.db.sql("SELECT * FROM econ.citation_counts").df().sort_values('weighted_citation_counts', ascending=False)
        sample = self.db.sql("SELECT * FROM econ.sample_names").df()
        dd = dict(zip(sample.author_id, sample.Group))
        print(f'{dd = }')
        df['group'] = [dd.get(aid, 'X') for aid in df.cited_id]
        df['pointsize'] = [0.01 if g == 'X' else 10 for g in df.group]
        df = df.sort_values(['group', 'pointsize'], ascending=[False, True])
        fig, ax = plt.subplots(1, 1)
        sns.scatterplot(df, x='citation_counts', y='weighted_citation_counts', hue='group', size='pointsize')
        ax.set_yscale('log')
        ax.set_xscale('log')
        ax.set_xlim((100, 50000))
        ax.set_ylim((100, 50000))
        hand, lab = ax.get_legend_handles_labels()
        lab[0] = 'Group'
        ax.legend()
        ax.legend(handles=list(hand[:4]), labels=list(lab[:4]), loc=2)
        ax.set_title(f'{citer_kind.title()}s citing {citer_kind}s')
        plt.show()
        return


In [51]:
def main():

    # cn = CoauthorshipNetwork()
    # cn.build_authorship_network()
    # cn.extract_graph()
    # cn.pagerank()

    kind = 'source'
    # kind = 'institution'
    cn = CitationNetwork(kind=kind)
    cn.build_base()
    # cn.report_base()
    # cn.build_citation_tables()
    # cn.report_citations()
    # cn.extract_citation_tables()
    return

In [52]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬──────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────